# 📊 Agregaciones con Pandas
### Python para Ciencia de Datos | UADE
#### Dataset: Airbnb Buenos Aires

---

En este notebook vamos a ver cómo **agrupar y resumir datos** usando `groupby` y funciones de agregación.

La idea central es:

```
df.groupby("columna_a_agrupar")["columna_a_medir"].funcion_de_agregacion()
```

| Función | Descripción |
|---|---|
| `count()` | Cantidad de valores no nulos |
| `sum()` | Suma de todos los valores |
| `mean()` | Promedio aritmético |
| `min()` / `max()` | Valor mínimo / máximo |
| `std()` | Desvío estándar |
| `agg([...])` | Varias funciones a la vez |


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("https://raw.githubusercontent.com/UADE-Python-Data-Science/583433_repo_oficial/refs/heads/main/Datasets/airbnb_jr_cleaned.csv")
# Vamos a usar el dataset airbnb_jr, pero la versión pre-procesada, sin nulos ni duplicados
print(f"Dataset: {df.shape[0]:,} filas × {df.shape[1]} columnas")
df.head(2)

---
## 1. Repaso de filtros

Antes de agregar, recordamos cómo filtrar filas.

In [ ]:
# Filtrar por barrio
df[df['barrio'] == 'Palermo'].head(3)

In [ ]:
# Convertir last_review a fecha
df["last_review"] = pd.to_datetime(df["last_review"], format="%Y-%m-%d", errors="coerce")

# Filtrar publicaciones del último año
df[df['last_review'].dt.year >= 2026].sort_values("last_review", ascending=False).head(3)

In [ ]:
# Filtro combinado: Palermo + último año
df[
    (df['last_review'].dt.year >= 2026) & (df['barrio'] == 'Palermo')
].sort_values("last_review", ascending=False).head(3)

---
## 2. Agregación con `groupby`

`groupby` divide el DataFrame en grupos según una columna y aplica una función sobre cada grupo.

```
df.groupby("columna")["métrica"].funcion()
```

Se sugiere complementar con la bibliografía [Wes McKinney](https://wesmckinney.com/book/data-aggregation)


### 2.1 Una columna de agrupamiento

In [ ]:
# Precio promedio por barrio — top 10
precio_por_barrio = (
    df.groupby("barrio", as_index=False)["price_usd"]
    .mean()
    .sort_values("price_usd", ascending=False)
    .head(10)
)
precio_por_barrio

In [ ]:
# Visualización: precio promedio por barrio (top 10)
plt.figure(figsize=(10, 5))
sns.barplot(data=precio_por_barrio, x="price_usd", y="barrio", color="steelblue")
plt.xlabel("Precio promedio (USD)")
plt.title("Top 10 barrios por precio promedio — Airbnb Buenos Aires")

### 2.2 Reviews por tipo de alojamiento

In [ ]:
# Total de reviews por tipo de alojamiento
reviews_por_tipo = (
    df.groupby("room_type", as_index=False)["number_reviews"]
    .sum()
    .sort_values("number_reviews", ascending=False)
)
reviews_por_tipo

In [ ]:
# Visualización: reviews por tipo de alojamiento
plt.figure(figsize=(10, 5))
sns.barplot(data=reviews_por_tipo, x="number_reviews", y="room_type", color="steelblue")
plt.xlabel("Número de reviews")
plt.title("Reviews por tipo de alojamiento — Airbnb Buenos Aires")

### 2.3 Dos columnas de agrupamiento

Podemos agrupar por más de una columna pasando una lista.

In [ ]:
# Reviews por barrio y tipo de alojamiento
df.groupby(["barrio", "room_type"], as_index=False)["number_reviews"].sum().head(10)

### 2.4 Agrupamiento por año

Extraemos el año de `last_review` y lo usamos como criterio de agrupamiento.

In [ ]:
# Crear columna de año
df["last_review_year"] = df["last_review"].dt.year

# Reviews acumuladas por año
reviews_por_anio = (
    df.groupby("last_review_year", as_index=False)["number_reviews"]
    .sum()
    .sort_values("last_review_year")
    .dropna()
)
reviews_por_anio

In [ ]:
# Visualización: evolución de reviews por año
plt.figure(figsize=(10, 5))
sns.barplot(data=reviews_por_anio, x="last_review_year", y="number_reviews", color="steelblue")
plt.xlabel("Número de reviews")
plt.title("Reviews por tipo año — Airbnb Buenos Aires")

---
## 3. Múltiples funciones con `agg`

Cuando queremos calcular varias métricas a la vez, usamos `.agg()`.

**Forma 1** — lista de funciones:
```python
df.groupby("barrio")["price_usd"].agg(["mean", "max", "min"])
```

**Forma 2** — diccionario con nombres personalizados (más legible):
```python
df.groupby("barrio").agg(
    promedio = ("price_usd", "mean"),
    maximo   = ("price_usd", "max"),
    minimo   = ("price_usd", "min")
)
```


In [ ]:
# Forma 1: lista de funciones
df.groupby("barrio", as_index=False)["price_usd"].agg(["mean", "max", "min"]).sort_values(
    "mean", ascending=False
).head(8)

In [ ]:
# Forma 2: nombres personalizados (más legible en informes)
resumen_barrio = (
    df.groupby("barrio", as_index=False)
    .agg(
        promedio = ("price_usd", "mean"),
        maximo   = ("price_usd", "max"),
        minimo   = ("price_usd", "min"),
        cantidad = ("price_usd", "count"),
    )
    .sort_values("promedio", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
resumen_barrio

In [ ]:
# Visualización: precio mínimo, promedio y máximo por barrio (top 5)
top8 = resumen_barrio.head(5)

# Transformar a formato largo para seaborn
top8_long = top8.melt(
    id_vars="barrio",
    value_vars=["minimo", "promedio", "maximo"],
    var_name="métrica",
    value_name="precio_usd"
)

sns.set_theme(style="whitegrid")

plt.figure(figsize=(12, 5))
sns.barplot(
    data=top8_long,
    x="barrio", y="precio_usd", hue="métrica",
    palette={"minimo": "#4CAF50", "promedio": "#2196F3", "maximo": "#E91E63"}
)
plt.xticks(rotation=0, ha="right")
plt.xlabel("")
plt.ylabel("Precio (USD)")
plt.title("Precio mínimo, promedio y máximo por barrio — Top 5")
plt.legend(title="Métrica")
plt.tight_layout()
plt.show()

---
## 🔑 Resumen

| Caso de uso | Código |
|---|---|
| Agrupar por una columna | `df.groupby("col")["métrica"].mean()` |
| Agrupar por varias columnas | `df.groupby(["col1", "col2"])["métrica"].sum()` |
| Varias métricas a la vez | `df.groupby("col").agg(["mean", "max", "min"])` |
| Nombres personalizados | `df.groupby("col").agg(prom=("col2","mean"), mx=("col2","max"))` |
| Sin índice jerárquico | `as_index=False` |

> 💡 **Próxima clase:** `merge` y `concat` — combinar múltiples DataFrames.
